# Week 1 lab — can one arrival rate describe a call centre?

**Getting the course materials.** We recommend forking the course repository on GitHub and working in your local clone. A fork lets you receive later course updates without putting your work in the course repository. After creating your fork, run:

```bash
git clone https://github.com/<your-account>/stochastic-modeling-simulation-ds.git
cd stochastic-modeling-simulation-ds
git remote add upstream https://github.com/NaitsabesMue/stochastic-modeling-simulation-ds.git
git pull --ff-only upstream main
```

Repeat `git pull --ff-only upstream main` when the lecturer announces an update. After pulling, make a working copy of this notebook and open the copy:

```bash
mkdir -p students/week01
cp notebooks/lecture1_poisson_warmup.ipynb students/week01/lecture1_poisson_warmup.ipynb
```

Work only in `students/week01/lecture1_poisson_warmup.ipynb`; leave the tracked notebook in `notebooks/` unchanged. If you pull a later course update, make a fresh copy only after saving your current work. The entire `students/` folder is ignored: never commit or push student work. VS Code with its Python/Jupyter extensions is convenient, but JupyterLab, Spyder, PyCharm, or another Python IDE is equally acceptable.

**Create a local Python environment.** From the repository folder, create one environment for this course and activate it. On macOS/Linux:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install jupyter ipykernel
```

On Windows PowerShell, use `py -m venv .venv`, then `.venv\Scripts\Activate.ps1` and the same two `python -m pip ...` commands. In VS Code or another IDE, select the interpreter inside `.venv`. The notebook itself uses only Python's standard library; `jupyter` and `ipykernel` are needed only to run it as a notebook. When finished, leave the environment with `deactivate`. The `.venv/` directory is local and ignored by Git.

Once you are ready let us start with the first question:

**Question.** Does one constant arrival rate look credible throughout a day? Start with the 96 counts from 1 November, then test a baseline that uses the same rate for every interval in the month. We will observe real aggregate counts, build a simple stochastic model, simulate it in plain Python, and check where it fails. The last step is to say what these counts cannot tell us about a staffing change.

**Data.** The course extract comes from the [Technion SEE Lab *AnonymousBank* data](https://seelab.net.technion.ac.il/data/), November 1999. Each row gives the number of non-PHANTOM calls entering the voice response unit (VRU) during one 15-minute interval. It contains no call, customer, or agent identifiers. Counts alone do not tell us when service began, how long it lasted, or how long anyone waited. Read the [dataset guide](../data/anonymousbank_data_guide.md) for the call path, outcome labels, raw fields, and limitations.

**Session routine.** The opening lecture and diagnostic happen on paper with student screens closed. During the lab you may use Python, the Internet and LLMs, and discuss with your group, but write your own code, answers and reasoning. Check any LLM output independently. Write short answers below each task or in new cells, and run the notebook from top to bottom before the exit quiz.

## 0. Warmup

1. **Python:** The counts below are strings, as they are when read from a CSV. Convert them to integers. Write an expression for the **number** of empty intervals, then calculate their **proportion**. Explain the denominator.
2. **Expected count:** A model predicts 2 entries *on average* per interval. Does seeing 5 in one interval disprove the model? Explain the difference between an expected count and an exact count.
3. **Model check:** The two integer lists below both have mean 2. Compute their population variances. Why is matching the mean alone insufficient when checking a demand model?
4. **Decision:** These counts record arrivals only. To estimate how one extra agent changes waiting, name three missing measurements and explain why each matters.

Add a code cell to check your Python and variance answers after writing your first attempt.


In [1]:
practice_text_counts = ["0", "1", "3", "1", "0", "5"]
equal_mean_a = [2, 2, 2, 2]
equal_mean_b = [0, 0, 4, 4]
print("CSV-like counts:", practice_text_counts)
print("Equal-mean lists:", equal_mean_a, equal_mean_b)


CSV-like counts: ['0', '1', '3', '1', '0', '5']
Equal-mean lists: [2, 2, 2, 2] [0, 0, 4, 4]


In [2]:
# 1. Convert strings to integers
counts = [int(x) for x in practice_text_counts]
print("Integer counts:", counts)

# 2. Number of empty intervals (value = 0)
num_empty = counts.count(0)
print("Number of empty intervals:", num_empty)

# 3. Proportion of empty intervals
proportion_empty = num_empty / len(counts)
print("Proportion of empty intervals:", proportion_empty)

Integer counts: [0, 1, 3, 1, 0, 5]
Number of empty intervals: 2
Proportion of empty intervals: 0.3333333333333333


In [4]:
import statistics
var_a = statistics.pvariance(equal_mean_a)
var_b = statistics.pvariance(equal_mean_b)
print("Variance A:", var_a)
print("Variance B:", var_b)

Variance A: 0
Variance B: 4


1.Service duration
2.Number of agents on duty
3.Abandonment / patience time

## 1. Observe: load the real counts

The CSV has three columns: `date` (local calendar day), `time` (start of a 15-minute interval), and `entries` (arrival count). The loader works when you launch Jupyter from the repository root or from `notebooks/`.

**Before running:** How many rows should November have if every day has 96 quarter-hour intervals?
30*96 = 2880 rows

In [5]:
import os
os.chdir(r"C:\Users\PR069694\Documents\stochastic-modeling-simulation-ds")
print("Working directory:", os.getcwd())

Working directory: C:\Users\PR069694\Documents\stochastic-modeling-simulation-ds


In [6]:
import csv
import math
import random
import statistics
from pathlib import Path

filename = "anonymousbank_nov1999_arrivals_15min.csv"
candidates = [Path.cwd() / "data" / filename, Path.cwd().parent / "data" / filename]
data_path = next((path for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError(
        f"Cannot find {filename} from {Path.cwd()}. Start Jupyter in the repository root or notebooks/."
    )

with data_path.open(newline="", encoding="utf-8") as source:
    reader = csv.DictReader(source)
    if reader.fieldnames != ["date", "time", "entries"]:
        raise ValueError(f"Unexpected CSV columns: {reader.fieldnames}")
    rows = [
        {"date": row["date"], "time": row["time"], "entries": int(row["entries"])}
        for row in reader
    ]

print(f"Loaded {len(rows)} intervals from {data_path}")
print("First eight intervals:")
for row in rows[:8]:
    print(f"{row['date']}  {row['time']}  {row['entries']:3d}")


Loaded 2880 intervals from C:\Users\PR069694\Documents\stochastic-modeling-simulation-ds\data\anonymousbank_nov1999_arrivals_15min.csv
First eight intervals:
1999-11-01  00:00    5
1999-11-01  00:15    2
1999-11-01  00:30    0
1999-11-01  00:45    1
1999-11-01  01:00    0
1999-11-01  01:15    3
1999-11-01  01:30    0
1999-11-01  01:45    0


Check the extract before fitting anything. A missing or duplicated interval would distort both the estimated rate and the time-of-day comparison. The assertions below are tailored to this teaching file: November 1999, 30 days, 96 quarter-hour rows per day, and 40,722 total entries. If you load another month or dataset, change the expected dates, slots and total accordingly.


In [7]:
dates = sorted({row["date"] for row in rows})
slots_by_date = {
    day: {row["time"] for row in rows if row["date"] == day}
    for day in dates
}
expected_dates = [f"1999-11-{day:02d}" for day in range(1, 31)]
expected_slots = {f"{hour:02d}:{minute:02d}" for hour in range(24) for minute in (0, 15, 30, 45)}
assert dates == expected_dates, "Expected every date in November 1999"
assert all(slots == expected_slots for slots in slots_by_date.values())
assert len(rows) == 30 * 96, "Expected 30 complete November days"
assert len(dates) == 30 and all(len(slots) == 96 for slots in slots_by_date.values())
assert len({(row["date"], row["time"]) for row in rows}) == len(rows)
assert rows == sorted(rows, key=lambda row: (row["date"], row["time"]))
assert all(row["entries"] >= 0 for row in rows)

observed = [row["entries"] for row in rows]
assert sum(observed) == 40722, "Total differs from the documented extract"
print(f"Dates: {dates[0]} to {dates[-1]}; total VRU entries: {sum(observed):,}")


Dates: 1999-11-01 to 1999-11-30; total VRU entries: 40,722


### First look: the 96 intervals on 1 November

This is the same day as the lecture plot. Each line below is one CSV row; each `#` represents one VRU entry in that 15-minute interval. There is no smoothing or averaging across intervals or days. These are aggregated call counts, not individual call timestamps.

Does a constant rate look credible for this day? Identify a quiet period and a busy period. Counts fluctuate even at a fixed rate, so a changing count alone does not establish a changing rate. To investigate whether the daily pattern also changes by weekday, we must compare multiple days.


In [8]:
first_day_rows = [row for row in rows if row["date"] == "1999-11-01"]
assert len(first_day_rows) == 96
print("1 November 1999: observed counts, one # per entry")
print("Time   Entries")
for row in first_day_rows:
    print(f"{row['time']}  {row['entries']:3d}  " + "#" * row["entries"])


1 November 1999: observed counts, one # per entry
Time   Entries
00:00    5  #####
00:15    2  ##
00:30    0  
00:45    1  #
01:00    0  
01:15    3  ###
01:30    0  
01:45    0  
02:00    0  
02:15    0  
02:30    0  
02:45    0  
03:00    0  
03:15    0  
03:30    0  
03:45    0  
04:00    0  
04:15    1  #
04:30    0  
04:45    0  
05:00    1  #
05:15    0  
05:30    0  
05:45    1  #
06:00    0  
06:15    3  ###
06:30    4  ####
06:45    5  #####
07:00   14  ##############
07:15   11  ###########
07:30   13  #############
07:45   19  ###################
08:00   18  ##################
08:15   35  ###################################
08:30   25  #########################
08:45   39  #######################################
09:00   40  ########################################
09:15   56  ########################################################
09:30   49  #################################################
09:45   51  ###################################################
10:00   49  #######

### Summarise all November intervals

Before running the next cell, predict whether a randomly chosen 15-minute interval from the **whole month** is likely to contain no entries. Then compare your prediction with the data.

The variance below uses divisor $n$, so it describes the observed set of bins. The ratio variance/mean is often called a *dispersion index*.


In [9]:
observed_mean = statistics.mean(observed)
observed_variance = statistics.pvariance(observed)
observed_zero_share = observed.count(0) / len(observed)

print(f"Mean entries per 15 minutes: {observed_mean:.2f}")
print(f"Variance across 15-minute bins: {observed_variance:.2f}")
print(f"Variance / mean: {observed_variance / observed_mean:.2f}")
print(f"Share of empty bins: {observed_zero_share:.1%}")
print(f"Largest count in one bin: {max(observed)}")


Mean entries per 15 minutes: 14.14
Variance across 15-minute bins: 209.30
Variance / mean: 14.80
Share of empty bins: 29.1%
Largest count in one bin: 78


The variance should be almost equal to the mean (only for poisson) but in this case it's 14.80 times more.

**Task 1.** What do the mean and empty-bin share say about a typical interval? Why might a whole-month average hide something important? Give two reasons, and label each one:

- **System or operational reason:** something about how the call centre actually behaves, such as quiet nights and busy mornings needing different staffing.
- **Data or measurement reason:** something about how the records were collected or summarised, such as 15-minute counts hiding the exact arrival times or VRU entries not being queue arrivals.

For each reason, write one sentence explaining how it could make the month-wide average misleading.


The average doesn't describe the period always well. almost 30 percent of intervals had zero.

- System/operational reason: The call centre is clearly busier at some times than others — nights are quiet, mornings get hectic. If you just take one average across the whole month and use that for staffing, you’ll have too few agents when it’s busy and too many sitting around at night.

- Data/measurement reason: Each row just tells you how many calls hit the VRU in a 15-minute window. You don’t know exactly when each call came in, or even if it actually reached an agent. So the count alone doesn’t really tell you how much work the agents had to deal with.

## 2. Model: a homogeneous Poisson arrival process

A first model assumes a **constant** arrival rate $r$ per minute. Under that model, arrivals in disjoint time intervals are independent; each 15-minute count has a Poisson distribution with mean $15r$. Its theoretical variance is also $15r$, and its probability of zero arrivals is $e^{-15r}$. Those are model claims to check, not properties guaranteed by the log.

Estimate $r$ from the whole-month average count. This deliberately simple estimate uses every 15-minute interval, including nights, busy hours and different days of the week. This is a deliberate baseline: one rate for all 2,880 intervals, rather than a rate fitted only to the day shown above. Keep track of units.


In [10]:
bin_minutes = 15
rate_per_minute = observed_mean / bin_minutes
model_zero_probability = math.exp(-observed_mean)

print(f"Estimated rate: {rate_per_minute:.3f} entries per minute")
print(f"Model probability of an empty 15-minute bin: {model_zero_probability:.4%}")
print(f"Observed share of empty bins: {observed_zero_share:.1%}")


Estimated rate: 0.943 entries per minute
Model probability of an empty 15-minute bin: 0.0001%
Observed share of empty bins: 29.1%


**Task 2.** Is the difference in empty-bin shares plausibly explained by sampling variation under the model? What does this comparison say about the *constant-rate assumption*?


The difference between the model probability of an empty bin and observed one is far too large to be explained by ordinary sampling variation. One constant arrival rate for the whole month is not a good fit.

### Simulate arrival times with plain Python

For a constant-rate Poisson process, successive times between arrivals are independent exponential random variables with mean $1/r$ minutes. Add each gap to a clock, then place the arrival in its 15-minute bin. We simulate as many bins as the real November extract.

Read the function before running it. In particular, explain why `int(clock // minutes_per_bin)` identifies the bin and why the loop stops at `horizon`.


In [11]:
def simulate_arrival_counts(n_bins, minutes_per_bin, rate_per_minute, rng):
    """Simulate constant-rate arrivals and return counts in consecutive bins."""
    if n_bins <= 0 or minutes_per_bin <= 0 or rate_per_minute <= 0:
        raise ValueError("Number of bins, bin width, and rate must be positive")
    counts = [0] * n_bins
    horizon = n_bins * minutes_per_bin
    clock = rng.expovariate(rate_per_minute)
    while clock < horizon:
        counts[int(clock // minutes_per_bin)] += 1
        clock += rng.expovariate(rate_per_minute)
    return counts

simulated = simulate_arrival_counts(
    len(observed), bin_minutes, rate_per_minute, random.Random(2026)
)
print(f"Simulated {sum(simulated):,} arrivals in {len(simulated)} bins")


Simulated 41,048 arrivals in 2880 bins


It uses a homogeneous Poisson process: the same estimated arrival rate applies throughout the whole simulation. So each 15-minute bin has a Poisson count with mean about 14.14.

### Check the simulator; then check the model

First compare simulated statistics with the model's predictions. The mean and variance should both be close to $15r$; agreement supports the implementation but is not a complete proof of correctness. Then compare with the observed counts to assess the model.

The table below reproduces the lecture's run: 2,880 intervals, seed `2026`, simulated mean 14.25, variance 14.52 and no empty intervals. Both variances use divisor $n$. The theoretical empty-interval probability is only about 0.000072%, so zero empty intervals in this run is unsurprising. Other seeds give different results.


In [12]:
simulated_mean = statistics.mean(simulated)
simulated_variance = statistics.pvariance(simulated)
simulated_zero_share = simulated.count(0) / len(simulated)

print(f"{'Quantity':<23} {'Model':>12} {'Simulated':>12} {'Observed':>12}")
print(f"{'Mean count':<23} {observed_mean:12.2f} {simulated_mean:12.2f} {observed_mean:12.2f}")
print(f"{'Variance of counts':<23} {observed_mean:12.2f} {simulated_variance:12.2f} {observed_variance:12.2f}")
print(f"{'Empty intervals':<23} {model_zero_probability:12.6%} {simulated_zero_share:12.1%} {observed_zero_share:12.1%}")
print(f"Simulated variance / mean: {simulated_variance / simulated_mean:.2f}")


Quantity                       Model    Simulated     Observed
Mean count                     14.14        14.25        14.14
Variance of counts             14.14        14.52       209.30
Empty intervals            0.000072%         0.0%        29.1%
Simulated variance / mean: 1.02


**Task 3.** Run the simulator with two other seeds. Which numbers change? Which conclusion does not? Do not try to tune the seed until the simulation resembles the data.


In [13]:
for seed in (7, 11):
    trial = simulate_arrival_counts(len(observed), bin_minutes, rate_per_minute, random.Random(seed))
    trial_mean = statistics.mean(trial)
    print(
        f"seed {seed:>2}: mean {trial_mean:5.2f}, "
        f"variance/mean {statistics.pvariance(trial) / trial_mean:5.2f}, "
        f"empty bins {trial.count(0) / len(trial):5.1%}"
    )


seed  7: mean 14.17, variance/mean  0.99, empty bins  0.0%
seed 11: mean 14.17, variance/mean  1.02, empty bins  0.0%


Both seeds give slightly different means and variance/mean ratios, but both stay close to the model’s predictions — around 14 and 1 respectively. Neither has any empty bins, which makes sense since the model only expects about 0.002 empty bins across 2,880 intervals. The takeaway doesn’t change: the simulator works as expected, but it still can’t reproduce the many empty bins we see in the real data

## 3. Validate: when does the model fail?

A model can match the overall average and still fail at the times when decisions matter. Compare the average count for each hour of day with the simulator. Unlike the single-day display above, this table averages counts. For 10:00, for example, it pools the 10:00, 10:15, 10:30 and 10:45 counts from each of the 30 days: 120 counts in total. The result is an average number of entries per 15-minute interval, not an hourly total or a moving average. The bars show the observed mean; each `#` is roughly two arrivals per 15-minute bin.


In [14]:
observed_by_hour = [[] for _ in range(24)]
simulated_by_hour = [[] for _ in range(24)]
for row, simulated_count in zip(rows, simulated):
    hour = int(row["time"][:2])
    observed_by_hour[hour].append(row["entries"])
    simulated_by_hour[hour].append(simulated_count)

print(f"{'Hour':<6} {'Observed':>9} {'Simulated':>10}  Observed profile")
for hour in range(24):
    observed_hour_mean = statistics.mean(observed_by_hour[hour])
    simulated_hour_mean = statistics.mean(simulated_by_hour[hour])
    print(
        f"{hour:02d}:00  {observed_hour_mean:9.1f} {simulated_hour_mean:10.1f}  "
        + "#" * round(observed_hour_mean / 2)
    )


Hour    Observed  Simulated  Observed profile
00:00        1.1       14.0  #
01:00        0.4       13.9  
02:00        0.3       13.9  
03:00        0.1       14.1  
04:00        0.2       13.9  
05:00        0.3       15.1  
06:00        1.3       14.0  #
07:00        8.1       14.4  ####
08:00       17.3       14.2  #########
09:00       30.7       14.4  ###############
10:00       32.3       14.5  ################
11:00       27.0       14.0  ##############
12:00       23.7       14.2  ############
13:00       23.5       13.6  ############
14:00       25.2       13.8  #############
15:00       26.1       14.3  #############
16:00       20.7       14.7  ##########
17:00       16.0       14.3  ########
18:00       16.5       14.6  ########
19:00       16.4       13.6  ########
20:00       13.8       14.8  #######
21:00       14.7       15.2  #######
22:00       13.9       14.5  #######
23:00        9.8       14.3  #####


In the real data, some hours are much quieter or busier than others. The table makes that difference visible and shows why one constant rate (the model itself) does not describe the daily pattern well.

**Task 4.** Find an hour when the model greatly overpredicts entries and one when it underpredicts them. Suppose you used the month-wide rate to plan a morning shift: what might go wrong? Suggest one better arrival model that still uses only these counts. State one further check you would make before trusting it.

Time-of-day variation is visible here, but it need not be the only source of mismatch. Next compare the same clock times on different days to investigate a possible weekly pattern. One day alone cannot establish a repeating weekly pattern. The documented normal working days here are Sunday–Thursday, so do not automatically use a Monday–Friday split. Estimate any proposed rates on earlier days and check them on later days; special dates and the 15-minute aggregation also deserve attention. 


The model overshoots at 3am and undershoots at 10am. If you used a single monthly average to plan a morning shift, you'd end up understaffed when it gets busy and overstaffed in the middle of the night. A better approach would be to use different arrival rates for different times of day, estimate them from the earlier days in the dataset, then see how well they predict the later days.

## 4. Decide: what is missing for a staffing question?

A manager asks: **“Will adding another agent reduce waiting times enough to justify the cost?”** The current extract contains arrival counts, but no service durations, agent schedules, queue entry/exit times, or abandonment records. It cannot answer that question on its own.

**Task 5.** List three additional measurements you would request. For each, say how it enters a queueing model or its validation. Then write a two-sentence recommendation to the manager: what the arrival analysis establishes, and what work is needed before making the staffing decision.


- 1.Service durations 
- 2.Agents on duty over time
- 3.Waiting and abandonment data

Recommendation: Demand shifts a lot throughout the day, so using a single constant rate just doesn’t cut it Before deciding whether to add an agent, collect service, staffing, and waiting data, then use them to build and validate a queueing model.

## 5. Use an LLM as a theory checker

### Reflect on the delegation
Before opening an LLM, write down the task you will delegate. Afterward, record: (1) the exact claim or code suggestion, (2) what was useful, (3) what you checked independently, (4) any error or missing assumption, and (5) what follow-up question made the answer deeper. A fluent answer is not evidence that it is correct. Do not include raw call records, identifiers or student work in the prompt.

First, answer this claim on your own: **“A count from a randomly selected one of two equally long Poisson periods is Poisson with the average rate.”** Here, **selecting one** means: flip a fair coin; on heads choose period A and observe one count from A; on tails choose period B and observe one count from B. You observe one count, from one period. You do not add the two counts.

If $A\sim\operatorname{Poisson}(1)$ and $B\sim\operatorname{Poisson}(3)$, the selected variable $X$ has $P(X=k)=\tfrac12P(A=k)+\tfrac12P(B=k)$. This is a **mixture**: first choose which distribution supplies the observation, then draw from that distribution. It is different from $A+B$, where both counts are observed and added.

Consider the fair choice between period A and period B. Is the selected count Poisson(2)?

Ask an LLM to prove or refute the claim, showing its steps. Check each step yourself by conditioning on which period was selected, or with a short Python experiment. Your check should give $E[X]=2$ and $\operatorname{Var}(X)=3$ for the selected count, while a Poisson(2) count has variance 2. Explain why **selecting one** count gives a mixture, whereas **adding** independent Poisson counts gives a sum. Record any correction you made to the LLM's argument or to your own.


In [17]:
def poisson_sample(lam, rng):
    # inverse transform sampling
    p = math.exp(-lam)
    F, k = p, 0
    u = rng.random()
    while u > F:
        k += 1
        p *= lam / k
        F += p
    return k

rng = random.Random(42)
samples = [poisson_sample(1 if rng.random()<0.5 else 3, rng) for _ in range(100000)]
print(f"E[X]   = {statistics.mean(samples):.3f}")      # ≈ 2
print(f"Var[X] = {statistics.pvariance(samples):.3f}") # ≈ 3

E[X]   = 2.006
Var[X] = 3.001


Not Poisson because the mean and the variance is not equal .

## Wrap-up before the exit quiz

Write down: (1) one observation from the real counts, (2) one assumption of the first model, (3) one concrete model failure, and (4) one thing you still cannot infer about staffing. 
